# NBL Cell Clustering

## Notebook Preferences

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format="retina"

## Libraries

In [ ]:
import buckaroo  # noqa: F401
import lamindb as ln
import spatialdata as sd
from nbl.util import DaskLocalCluster
import nbl
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("future.infer_string", False)
pd.set_option("mode.copy_on_write", True)

In [ ]:
# ln.context.uid = "mqEhM9LGLNtM"
# ln.settings.transform.version = "1"
# ln.settings.sync_git_repo = "https://github.com/karadavis-lab/nbl.git"
#  ln.track()
# run.transform

In [ ]:
ln.setup.settings.instance

In [ ]:
cluster = DaskLocalCluster(n_workers=10, threads_per_worker=1)
cluster(open_dashboard=True)

## Get `FeatureSet` from Immune Markers Feature Set

In [ ]:
immune_infiltrate_markers = nbl.ln.cell_marker_set_catalog("immune_infiltrate", "featureset")

In [ ]:
nbl_sdata = sd.read_zarr(ln.Artifact.filter(key__contains="nbl.zarr").one().path)

In [ ]:
nbl_wc = nbl_sdata.tables["whole_cell"]

nbl_wc_NBL = nbl_wc[nbl_wc.obs["pixie_cluster"] == "NBL_cell"]

In [ ]:
def func(pct, allvals):
    """Returns a string with the percentage and absolute number of cells.

    Parameters
    ----------
    pct : _type_
        _description_
    allvals : _type_
        _description_

    Returns
    -------
    _type_
        _description_
    """
    absolute = int(np.round(pct / 100.0 * np.sum(allvals)))
    return f"{pct:.1f}% ({absolute:d} Cells)"

In [ ]:
immune_infiltrate_markers.members.filter(name__contains="CD45").one().name

In [ ]:
for i in [immune_infiltrate_markers.members.filter(name__contains="CD45").one().name]:
    marker = i
    nbl_wc_NBL__90 = nbl.tl.quantile(adata=nbl_wc_NBL, var=marker, q=0.90, filter_adata=True)
    nbl_wc_NBL__95 = nbl.tl.quantile(adata=nbl_wc_NBL, var=marker, q=0.95, filter_adata=True)
    nbl_wc_NBL__99 = nbl.tl.quantile(adata=nbl_wc_NBL, var=marker, q=0.99, filter_adata=True)

    nbl_wc_NBL.obs[f"{marker}_90"] = nbl_wc_NBL.obs_names.isin(nbl_wc_NBL__90.obs_names)
    nbl_wc_NBL.obs[f"{marker}_95"] = nbl_wc_NBL.obs_names.isin(nbl_wc_NBL__95.obs_names)
    nbl_wc_NBL.obs[f"{marker}_99"] = nbl_wc_NBL.obs_names.isin(nbl_wc_NBL__99.obs_names)

    n__90 = sum(nbl_wc_NBL.obs[f"{marker}_90"]) - (
        sum(nbl_wc_NBL.obs[f"{marker}_95"]) + sum(nbl_wc_NBL.obs[f"{marker}_99"])
    )
    n__95 = sum(nbl_wc_NBL.obs[f"{marker}_95"]) - sum(nbl_wc_NBL.obs[f"{marker}_99"])
    n__99: int = sum(nbl_wc_NBL.obs[f"{marker}_99"])

    n_nbl_cells_rest = nbl_wc_NBL.n_obs - n__90 - n__95 - n__99

    sizes = [n__90, n__95, n__99, n_nbl_cells_rest]
    labels = [f"{marker}_90", f"{marker}_95", f"{marker}_99", "NBL Cells Rest"]

    fig, ax = plt.subplots(figsize=(16, 8), subplot_kw={"aspect": "equal"})

    wedges, texts = ax.pie(sizes, startangle=-40)

    bbox_props = {"boxstyle": "square,pad=0.3", "fc": "w", "ec": "k", "lw": 0.72}
    kw = {"arrowprops": {"arrowstyle": "-"}, "bbox": bbox_props, "zorder": 0, "va": "center"}

    for i, p in enumerate(wedges):
        ang = (p.theta2 - p.theta1) / 2.0 + p.theta1
        y = np.sin(np.deg2rad(ang))
        x = np.cos(np.deg2rad(ang))
        horizontalalignment = {-1: "right", 1: "left"}[int(np.sign(x))]
        connectionstyle = f"angle,angleA=0,angleB={ang}"
        kw["arrowprops"].update({"connectionstyle": connectionstyle})
        ax.annotate(
            f"{labels[i]} \n {func(sizes[i] / sum(sizes) * 100, sizes)}",
            xy=(x, y),
            xytext=(1.35 * np.sign(x), 1.4 * y),
            horizontalalignment=horizontalalignment,
            **kw,
        )

    ax.set_title("NBL Cells Distribution")

    plt.show()

In [ ]:
import seaborn.objects as so

In [ ]:
so.Plot(nbl_wc_NBL.obs).add(so.Bar(width=10), so.Hist(stat="count")).pair(x=["CD45_90", "CD45_95", "CD45_99"]).label(
    x0=r"CD45 $90^{th}$", x1=r"CD45 $95^{th}$", x2=r"CD45 $99^{th}$"
).share(x=True, y=True).show()

In [ ]:
nbl_wc_NBL.uns["quantiles"]

In [ ]:
nbl_wc_NBL.to_df()

In [ ]:
nbl_markers = nbl.ln.cell_marker_set_catalog("neuroblastoma", "names")

In [ ]:
import scanpy as sc

In [ ]:
import pandas as pd


def combine_columns(row) -> str:
    """Combines columns into a single column based on the values in the columns.

    Parameters
    ----------
    row : pd.Series
        The row of the DataFrame.

    Returns
    -------
    str
        The combined column.
    """
    if row["CD45_99"]:
        return "Group_99"
    elif row["CD45_95"]:
        return "Group_95"
    elif row["CD45_90"]:
        return "Group_90"
    else:
        return "Rest"


# Apply the function to combine columns
nbl_wc_NBL.obs["Group"] = nbl_wc_NBL.obs.apply(combine_columns, axis=1)

# Check the new column
print(nbl_wc_NBL.obs["Group"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), dpi=300, layout="constrained", sharex=True, sharey=True)
fig.suptitle(t="NBL Markers Distribution -- Grouped by CD45 Expression Quantiles (Exclusive)")
sc.pl.stacked_violin(adata=nbl_wc_NBL, var_names=nbl_markers, groupby="Group", ax=ax)
fig

In [ ]:
ln.finish()